TASK 3: CITY, LATITUDE AND LONGITUDE

In [ ]:
import math
from random import choice

# Declaring list of example cities with their coordinates
cities = {
'Bergen': [60.364098, 5.404422, 15],
'Oslo': [59.926959, 10.779454, 10],
'Trondheim': [63.433654, 10.390277, 5]
}

# Declaring list of example places of interest (poi) with their coordinates
poi = {
(63.426806, 10.396712): ['Nidarosdommen', 4.5],
(59.926699, 10.701075): ['Vigelandsparken', 5],
(59.907704, 10.753163): ['Operahuset', 4],
(60.397561, 5.322833): ['Bryggen', 4.5],
(60.387639, 5.321623): ['Universitetmuseet', 5],
(60.342901, 5.336796): ['Gamlehaugen', 4]
}

# Declaring latitude and longitude per task description
def_latitude = 111
def_longitude = 68

# Reusable distance function
def calc_dist(lat1, lon1, lat2, lon2 ):
        # converting to km using the predefined values
        lat_diff = (lat1 - lat2) * def_latitude
        lon_diff = (lon1 - lon2) * def_longitude
        # euclidean (pythagorean)
        return math.sqrt(lat_diff ** 2 + lon_diff ** 2)


In [ ]:
# TASK A, Functions that find the nearest city and POI

def find_nearest_city(latitude, longitude):
    # starting with infinity
    min_dist = float('inf')
    nearest_city = None
    # traverse item, set names to value
    for city_name, (city_latitude, city_longitude, city_size) in cities.items():
        dist = calc_dist(latitude, longitude, city_latitude, city_longitude)
        if dist < min_dist:
            # set
            min_dist = dist
            nearest_city = city_name
    return nearest_city, min_dist

def find_nearest_poi(latitude, longitude):
     # starting with infinity
    min_dist = float('inf')
    nearest_poi = None
     # traverse item differently from cities item, by coordinates
    for (poi_latitude, poi_longitude), (poi_name, poi_rating) in poi.items():
        dist = calc_dist(latitude, longitude, poi_latitude, poi_longitude)
        if dist < min_dist:
            # set
            min_dist = dist
            nearest_poi = poi_name
    return nearest_poi, min_dist

def get_input():
    while True:
        print("1: find nearest cities\n")
        print("2: find nearest poi\n")
        print("3: find nearest cities & poi\n")
        print("4: quit\n")

        user_choice = input("Enter your choice: \n")

        if user_choice == "1":
            input_lon = float(input('Enter city longitude: \n'))
            input_lat = float(input('Enter city latitude: \n'))
            # capture
            city_item, dist = find_nearest_city(input_lat, input_lon)
            print(f"Nearest city: {city_item} (distance: {dist:.2f} km)\n")

        elif user_choice == "2":
            input_lon = float(input('Enter poi longitude: \n'))
            input_lat = float(input('Enter poi latitude: \n'))
            # capture
            poi_item, dist = find_nearest_poi(input_lat, input_lon)
            print(f"Nearest poi: {poi_item} (distance: {dist:.2f} km)\n")

        elif user_choice == "3":
            input_lon = float(input('Enter city & poi longitude: \n'))
            input_lat = float(input('Enter city & poi latitude: \n'))

            city, dist = find_nearest_city(input_lat, input_lon)
            print(f"Nearest city: {city} (distance: {dist:.2f} km)\n")
            poi_item, dist = find_nearest_poi(input_lat, input_lon)
            print(f"Nearest poi: {poi_item} (distance: {dist:.2f} km)\n")

        elif user_choice == "4":
            print("Quit\n")
            break

        else:
            print("Invalid input\n")

# RUNNING PROGRAM
get_input()


In [ ]:
# TASK B, Compute city boundaries

def compute_city_boundaries(city):
    # create empty list
    bounds = {}
    for city_name, (city_latitude, city_longitude, city_size) in city.items():
        # from km to degrees
        lat_offset = city_size / def_latitude
        lon_offset = city_size / def_longitude

        # setting boundaries
        bounds[city_name] = {
            "north": city_latitude + lat_offset,
            "south": city_latitude - lat_offset,
            "east": city_longitude + lon_offset,
            "west": city_longitude - lon_offset
        }
    return bounds

# Calling function
city_bounds = compute_city_boundaries(cities)
print(city_bounds)


In [ ]:
# TASK C, Assign POIs to cities

def assign_city_pois(pois_list, city_boundaries):
    # create empty list
    city_pois = {city_name: [] for city_name in city_boundaries.keys()}
    # traverse each poi
    for (poi_latitude, poi_longitude), (poi_name, poi_rating) in pois_list.items():
        # traverse through each city
        for city_name, bounds in city_boundaries.items():
            # check city bounds
            if bounds['south'] <= poi_latitude <= bounds['north'] and bounds['west'] <= poi_longitude <= bounds['east']:
                # add coordinates for reusability in task D & E
                city_pois[city_name].append(((poi_latitude, poi_longitude), poi_name))
                break
    return city_pois

# Calling function
pois_by_city = assign_city_pois(poi, city_bounds)
print(pois_by_city)


In [ ]:
# TASK D, Compute average POI rating per city

def average_poi_rating(pois_by_city ,pois_original):
    # create empty list
    avrg_poi = {}
    # traverse through each city
    for city_name, poi_data in pois_by_city.items():
        # validation, no poi = empty list, reasoning we are doing division by variable data
        if not poi_data:
            avrg_poi[city_name] = 0
        else:
            # direct lookup
            total_rating = sum(pois_original[coords][1] for coords, name in poi_data)
            # simple calculation for getting average rating
            avrg_poi[city_name] = total_rating / len(poi_data)

    return  avrg_poi

average_rating = average_poi_rating(pois_by_city, poi)
print(average_rating)


In [ ]:
# TASK E, Order POIs by distance from the city center

def poi_dist(pois_city, pois_original):
    # create empty list
    poi_sorted_by_dist = {}
    # traverse through each city
    for city_name, poi_data in pois_city.items():
         # unpack 3 values
        city_lat, city_lon, _ = cities[city_name]
        poi_distances = []
        # traverse each poi in city
        for (poi_lat, poi_lon), poi_name in poi_data:
            # calculate distance from center
            dist = calc_dist(city_lat, city_lon, poi_lat, poi_lon)
            # store as tuple
            poi_distances.append((dist, poi_name))
        # sort by distance, takes each element as 'x' and uses the first index
        poi_distances.sort(key=lambda x: x[0])
        # reverse tuple and round distance
        poi_sorted_by_dist[city_name] = [(name, round(dist, 2)) for dist, name in poi_distances]

    return poi_sorted_by_dist

pois_ordered = poi_dist(pois_by_city, poi)
print(pois_ordered)
